In [1]:
!pip install -q torch torchvision transformers datasets pillow pandas scikit-learn tqdm huggingface_hub matplotlib

In [2]:
import os
import json
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from PIL import Image
from tqdm.auto import tqdm
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from transformers import CLIPModel, CLIPImageProcessor, get_cosine_schedule_with_warmup

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

_Config_

In [3]:
DATASET_NAME = "ashraq/fashion-product-images-small"
MODEL_NAME = "openai/clip-vit-base-patch32"
TASKS = [
    "gender",
    "masterCategory",
    "subCategory",
    "articleType",
    "baseColour",
    "season",
    "usage"
]

SEED = 42
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

BATCH_SIZE = 32
EPOCHS = 5

HEAD_LR = 3e-4
BACKBONE_LR = 1e-5
WEIGHT_DECAY = 1e-2

HIDDEN_DIM = 512
DROPOUT = 0.20

UNFREEZE_LAST_N_VISION_LAYERS = 2

USE_CLASS_WEIGHTS = True
USE_AMP = True

MAX_GRAD_NORM = 1.0
EARLY_STOPPING_PATIENCE = 2
NUM_WORKERS = 2
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [4]:
OUTPUT_DIR = Path("artifacts/models/autocatalogai_clip")
EVAL_DIR = Path("artifacts/evaluation")
PLOT_DIR = Path("artifacts/plots")
PROCESSED_DIR = Path("data/processed")

for directory in [OUTPUT_DIR, EVAL_DIR, PLOT_DIR, PROCESSED_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("Device:", DEVICE)
print("Model:", MODEL_NAME)

Device: cuda
Model: openai/clip-vit-base-patch32
